**Ingest circuits.csv file**
1. Read file using spark dataframe reader API
2.add Metadata Columns 
-   source file
- ingestion Timestamp
3. write to bonze delta table

In [0]:
%run ../00.Common/01.environment-config


In [0]:
%run ../00.Common/02.bronze-helpers


In [0]:
source_file = f"{landing_folder_path}/circuits.csv"
table_name = f"{catalog_nameone}.{bronze_schema}.circuits"

display(table_name)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuit_schema = StructType([
    StructField('circuitId', StringType()),
    StructField('url', StringType()),
    StructField('circuitName', StringType()),
    StructField('lat', DoubleType()),
    StructField('long', DoubleType()),
    StructField('locality', StringType()),
    StructField('country', StringType())

])
circuits_df =(
     spark.read.format("csv")
          .option("header", "true")
          .option("mode", "FAILFAST")
          .schema(circuit_schema)
          .load(source_file)
)


circuits_final_df = add_ingestion_metadata(circuits_df)

display(circuits_final_df)

In [0]:
circuits_final_df.write.format('delta').mode('overwrite').saveAsTable(table_name)